In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Data Overview

## NIST Photovoltaic Dataset

Dataset Source:  
https://pvdata.nist.gov/

---

## General Information

| Attribute | Description |
|---|---|
| **Location** | Maryland, United States |
| **Data Type** | Real-world photovoltaic system measurements |
| **Time Granularity** | 1-minute intervals |
| **Years Used** | 2015 until 2018 |

---

## Measurement Locations

| Location | Description |
|---|---|
| **Ground** | Solar panels installed on ground-mounted structures (fixed tilt ≈ 20°) |
| **Roof** | Solar panels installed on building rooftops |
| **Canopy** | Solar panels installed as parking canopy/shading structures |
| **ws_1, ws_2** | Independent weather stations collecting meteorological measurements |

---

## Dataset Size

| Dataset | Years Included | Total Rows |
|---|---|---|
| **Canopy Data** | 2015, 2016, 2017, 2018 | 2,102,400 |
| **Ground Data** | 2015, 2016, 2017, 2018 | 2,102,400 |
| **Roof Data** | 2015, 2016, 2017, 2018 | 1,706,400 |

## Dataset Structure Verification and File Counting

The dataset directory structure is scanned to verify that all expected CSV files are present for each month and year.

The dataset is organized hierarchically as:

```text
NIST/
│
├── onemin-Canopy-2015/
│   ├── 01/
│   ├── 02/
│   └── ...
│
├── onemin-Canopy-2016/
│   ├── 01/
│   ├── 02/
│   └── ...
```

Each monthly folder contains daily CSV files, where:

- **1 CSV file = 1 day of measurements**
- Therefore, the number of files in a month should normally match the number of days in that month:
  - 31 files for months with 31 days
  - 30 files for months with 30 days
  - 28 or 29 files for February

The code iterates through all dataset folders and monthly subfolders, counts the number of `.csv` files, and prints the results. This acts as a preliminary data integrity check before preprocessing and model development.

This verification step is important because it helps identify:

- Missing files
- Incomplete downloads
- Corrupted or partially collected data
- Irregularities in the dataset structure

For example:

```text
onemin-Roof-2018 | Month 03 → 18 files
```

indicates that March 2018 contains fewer files than expected, suggesting incomplete data for that period.

The file counts are also stored in a nested Python dictionary (`file_counts`) for later analysis if needed.

In [ ]:
import os

base_path = "/content/drive/MyDrive/NIST"

file_counts = {}

for folder in sorted(os.listdir(base_path)):
    folder_path = os.path.join(base_path, folder)

    if os.path.isdir(folder_path):
        file_counts[folder] = {}

        for month in sorted(os.listdir(folder_path)):
            month_path = os.path.join(folder_path, month)

            if os.path.isdir(month_path):
                csv_files = [f for f in os.listdir(month_path) if f.endswith(".csv")]
                count = len(csv_files)

                file_counts[folder][month] = count

                print(f"{folder} | Month {month} → {count} files")


onemin-Canopy-2015 | Month 01 → 31 files
onemin-Canopy-2015 | Month 02 → 28 files
onemin-Canopy-2015 | Month 03 → 31 files
onemin-Canopy-2015 | Month 04 → 30 files
onemin-Canopy-2015 | Month 05 → 31 files
onemin-Canopy-2015 | Month 06 → 30 files
onemin-Canopy-2015 | Month 07 → 31 files
onemin-Canopy-2015 | Month 08 → 31 files
onemin-Canopy-2015 | Month 09 → 30 files
onemin-Canopy-2015 | Month 10 → 31 files
onemin-Canopy-2015 | Month 11 → 30 files
onemin-Canopy-2015 | Month 12 → 31 files
onemin-Canopy-2016 | Month 01 → 31 files
onemin-Canopy-2016 | Month 02 → 29 files
onemin-Canopy-2016 | Month 03 → 31 files
onemin-Canopy-2016 | Month 04 → 30 files
onemin-Canopy-2016 | Month 05 → 31 files
onemin-Canopy-2016 | Month 06 → 30 files
onemin-Canopy-2016 | Month 07 → 31 files
onemin-Canopy-2016 | Month 08 → 31 files
onemin-Canopy-2016 | Month 09 → 30 files
onemin-Canopy-2016 | Month 10 → 31 files
onemin-Canopy-2016 | Month 11 → 30 files
onemin-Canopy-2016 | Month 12 → 31 files
onemin-Canopy-20

## Merging Daily CSV Files

This step combines all daily CSV files for each dataset folder into a single unified CSV file.

The code:

- Iterates through all installation/year folders
- Reads every daily `.csv` file from each monthly subfolder
- Concatenates them into one DataFrame
- Saves the merged dataset into:
  ```text
  /content/drive/MyDrive/NIST_processed
  ```

This preprocessing step simplifies later analysis and machine learning workflows by converting thousands of small daily files into a smaller number of continuous datasets.

The output also helps verify dataset completeness. Since the data has a 1-minute granularity:

- A normal year should contain:
  ```text
  24 × 60 × 365 = 525,600 records
  ```
- A leap year should contain:
  ```text
  24 × 60 × 366 = 527,040 records
  ```

The Canopy datasets match these expected values for 2015–2017, while the 2018 dataset contains fewer records, indicating missing data. Similar inconsistencies are also observed in some Ground and Roof datasets.


Memory is also managed efficiently by saving each merged dataset immediately and deleting it from memory after processing.

In [ ]:
import os

base_path = '/content/drive/MyDrive/NIST'
output_path = '/content/drive/MyDrive/NIST_processed'

os.makedirs(output_path, exist_ok=True)

for folder in sorted(os.listdir(base_path)):
    folder_path = os.path.join(base_path, folder)

    if os.path.isdir(folder_path):
        combined_df = None

        for month in sorted(os.listdir(folder_path)):
            month_path = os.path.join(folder_path, month)

            if os.path.isdir(month_path):
                for file in sorted(os.listdir(month_path)):
                    if file.endswith(".csv"):
                        file_path = os.path.join(month_path, file)

                        df = pd.read_csv(file_path)

                        # Incremental concat (memory safer)
                        if combined_df is None:
                            combined_df = df
                        else:
                            combined_df = pd.concat([combined_df, df], ignore_index=True)

        # Save to disk immediately
        save_path = os.path.join(output_path, f"{folder}.csv")
        combined_df.to_csv(save_path, index=False)

        print(f"Saved: {folder} → {combined_df.shape}")

        # 🔥 CRITICAL: free memory
        del combined_df

Saved: onemin-Canopy-2015 → (525600, 102)
Saved: onemin-Canopy-2016 → (527040, 102)
Saved: onemin-Canopy-2017 → (525600, 102)
Saved: onemin-Canopy-2018 → (525107, 102)
Saved: onemin-Ground-2015 → (525588, 100)
Saved: onemin-Ground-2016 → (527040, 100)
Saved: onemin-Ground-2017 → (525595, 100)
Saved: onemin-Ground-2018 → (525587, 100)
Saved: onemin-Roof-2015 → (523450, 123)
Saved: onemin-Roof-2016 → (521270, 123)
Saved: onemin-Roof-2017 → (525599, 123)
Saved: onemin-Roof-2018 → (110838, 123)
